In [3]:
import pandas as pd
import pysam
import os
from Bio.Seq import Seq
import collections
from tqdm import tqdm
import collections
import ast
import gzip
from Bio import SeqIO

In [4]:
def add_feature(existing_features, new_feature):
    return existing_features + (new_feature,)

In [5]:
phylogeneticOrder=()
phylDF = pd.read_csv("/LeeLab/HPRC/chromosomeY/HPRC_HGSVC3_sample_annotations_20March2025.txt",sep='\t')
for row in phylDF.index:
    if '/' in phylDF.at[row,'sample']:
        for sample in phylDF.at[row,'sample'].split("/"):
             phylogeneticOrder = add_feature(phylogeneticOrder, sample)
    else:
        phylogeneticOrder = add_feature(phylogeneticOrder, phylDF.at[row,'sample'])

haplogroups={}
for row in phylDF.index:
    if '/' in phylDF.at[row,'sample']:
        for sample in phylDF.at[row,'sample'].split("/"):
            haplogroups[sample]=phylDF.at[row,'haplogroup_ISOGG_v15.73']
    else:
        haplogroups[phylDF.at[row,'sample']]=phylDF.at[row,'haplogroup_ISOGG_v15.73']

haplogroups['NA12877']='R1b1a1b1a1a2e2'
haplogroups['NA12882']='R1b1a1b1a1a2e2'
haplogroups['NA12883']='R1b1a1b1a1a2e2'
haplogroups['NA12884']='R1b1a1b1a1a2e2'
haplogroups['NA12886']='R1b1a1b1a1a2e2'
haplogroups['200080']='R1b1a1b1a1a1c2b2b1a'
haplogroups['200084']='R1b1a1b1a1a1c2b2b1a'
haplogroups['200085']='R1b1a1b1a1a1c2b2b1a'

In [6]:
def generate_kmers(sequence, kmer_length):
    kmers = [sequence[i:i+kmer_length] for i in range(0, len(sequence) - kmer_length + 1)]
    return kmers

In [7]:
colorDF = pd.read_csv("/LeeLab/HPRC/chromosomeY/GRCh38-Y_regions_repeat-details_19Apr25.bed.txt", sep='\t')
colorList={x.strip():{} for x in colorDF['seqclass']}
for row in colorDF.index:
    if str(colorDF.at[row,'seqclass']).strip() in colorList:
        colorList[colorDF.at[row,'seqclass'].strip()]['Start']=int(colorDF.at[row,'start'])
        colorList[colorDF.at[row,'seqclass'].strip()]['End']=int(colorDF.at[row,'end'])
    else:
        continue

In [8]:
colorhg38Orientation={
    'blue1':'C',
    'blue2':'+',
    'blue3':'C',
    'blue4':'+',
    'teal1':'C',
    'teal2':'+',
    'green1':'C',
    'green2':'C',
    'green3':'+',
    'red1':'C',
    'red2':'+',
    'red3':'C',
    'red4':'+',
    'gray1':'C',
    'gray2':'+',
    'yellow1':'C',
    'yellow2':'+',
    'spacer1':'+',
    'spacer2':'+',
    'spacer3':'+'}

In [9]:
Colorkmers={x:{} for x in colorList.keys()}
for refinedColor in tqdm(colorList.keys()):
    
    if 'plus' in refinedColor or 'IR1' in refinedColor:
        color = refinedColor.split("-")[0]
    else:
        color=refinedColor
        

    if colorhg38Orientation[color] == '+':
        sequence = str(pysam.faidx('/LeeLab/referenceSequences/GRCh38_chrY.fasta', 'chrY:'+str(colorList[refinedColor]['Start'])+"-"+str(colorList[refinedColor]['End'])).split()[1:][0]).upper()
        colorkmers = generate_kmers(sequence, 31)
        forwardSet = set(x for x in collections.Counter(colorkmers))

        sequence = pysam.faidx('/LeeLab/referenceSequences/GRCh38_chrY.fasta', 'chrY:'+str(colorList[refinedColor]['Start'])+"-"+str(colorList[refinedColor]['End'])).split()[1:][0]
        revSeq = str(Seq(sequence).reverse_complement()).upper()
        colorkmersRev = generate_kmers(revSeq, 31)
        reverseSet = set(x for x in collections.Counter(colorkmersRev))
    
    else:
        sequence = str(pysam.faidx('/LeeLab/referenceSequences/GRCh38_chrY.fasta', 'chrY:'+str(colorList[refinedColor]['Start'])+"-"+str(colorList[refinedColor]['End'])).split()[1:][0]).upper()
        colorkmers = generate_kmers(sequence, 31)
        reverseSet = set(x for x in collections.Counter(colorkmers))

        sequence = pysam.faidx('/LeeLab/referenceSequences/GRCh38_chrY.fasta', 'chrY:'+str(colorList[refinedColor]['Start'])+"-"+str(colorList[refinedColor]['End'])).split()[1:][0]
        revSeq = str(Seq(sequence).reverse_complement()).upper()
        colorkmersRev = generate_kmers(revSeq, 31)
        forwardSet = set(x for x in collections.Counter(colorkmersRev))
        

    for kmer in forwardSet:
       Colorkmers[refinedColor][kmer]='sense'

    for kmer in reverseSet:
        if kmer in Colorkmers[refinedColor]:
            Colorkmers[refinedColor][kmer]='both' 
        else:
            Colorkmers[refinedColor][kmer]='antisense'    


100%|█████████████████████████████████████████████████████████████████████████| 29/29 [00:02<00:00, 11.53it/s]


In [10]:
from typing import List, Dict, Tuple, Optional

def classify_sequence_with_kmers_fast(
    dna_seq: str,
    kmer_dict: Dict[str, Dict[str, str]],
    window_size: int,
    allowed_colors: List[str],
    region_start: int,
    region_end: int
) -> Tuple[
    List[Tuple[int, int, str, str]],  
    int,  
    int,  
    int,  
    int   
]:
    """
    Slide non-overlapping windows of size `window_size` across `dna_seq`
    (which corresponds to [region_start, region_end) in the genome).
    Return:
      1) windows:  [(abs_start, abs_end, colors, orient), ...]
      2) refined_start: region_start + unassigned_start_bases
      3) refined_end:   region_end   - unassigned_end_bases
      4) unassigned_start_bases
      5) unassigned_end_bases
    """

    seq_len = len(dna_seq)
    kmer_lookup: Dict[str, Tuple[str, str]] = {}
    for color in allowed_colors:
        for kmer, orient in kmer_dict.get(color, {}).items():
            if kmer in kmer_lookup:
                prev_cols, prev_orient = kmer_lookup[kmer]
                cols = set(prev_cols.split("_"))
                cols.add(color)
                merged_cols = "_".join(sorted(cols))
                merged_orient = "both" if prev_orient != orient else orient
                kmer_lookup[kmer] = (merged_cols, merged_orient)
            else:
                kmer_lookup[kmer] = (color, orient)

    kmer_lengths = sorted({len(k) for k in kmer_lookup}, reverse=True)

    results: List[Tuple[int, int, str, str]] = []
    i = 0
    while i + window_size <= seq_len:
        window = dna_seq[i : i + window_size]
        match: Optional[Tuple[str, str]] = None
        for klen in kmer_lengths:
            for j in range(window_size - klen + 1):
                sub = window[j : j + klen]
                if sub in kmer_lookup:
                    match = kmer_lookup[sub]
                    break
            if match:
                break
    
        abs_start = region_start + i
        abs_end   = region_start + i + window_size
    
        if match:
            cols, orient = match
            step = window_size
        else:
            cols, orient = "", ""
            step = 1
    
        results.append((abs_start, abs_end, cols, orient))
        i += step


    if i < seq_len:
        window = dna_seq[i:]
        match = None
        for klen in kmer_lengths:
            for j in range(len(window) - klen + 1):
                sub = window[j : j + klen]
                if sub in kmer_lookup:
                    match = kmer_lookup[sub]
                    break
            if match:
                break

        abs_start = region_start + i
        abs_end   = region_end
        if match:
            cols, orient = match
        else:
            cols, orient = "", ""
        results.append((abs_start, abs_end, cols, orient))


    first_idx = next((idx for idx, (_, _, cols, _) in enumerate(results) if cols), None)
    if first_idx is None:
        unassigned_start_bases = seq_len
        unassigned_end_bases   = seq_len
        refined_start = region_start
        refined_end   = region_end
    else:
        unassigned_start_bases = results[first_idx][0] - region_start
        refined_start = region_start + unassigned_start_bases
        last_idx = len(results) - 1 - next(
            idx for idx, (_, _, cols, _) in enumerate(reversed(results)) if cols
        )
        unassigned_end_bases = region_end - results[last_idx][1]
        refined_end = region_end - unassigned_end_bases

    return results, refined_start, refined_end, unassigned_start_bases, unassigned_end_bases

In [11]:
k = 31
assemblyDirectory = '/LeeLab/Assemblies/HPRC_Release2/chrY_assemblies/'
directory = '/LeeLab/HPRC/chromosomeY/Data/ColorBlockDataframes/allInfo/'

for sample in tqdm(os.listdir(directory)):
    if '.csv' in str(sample):

        if '.csv' in sample:

            NgapList=[]
            filename = sample.split(".csv")[0]
            sampleName = str(sample.split("_")[0])
    
            df = pd.read_csv(directory+sample).drop(columns=['Unnamed: 0'])
            df['RefinedStart']='temp'
            df['RefinedEnd']='temp'
            df['RefinedClassification']='temp'
            df['RefinedUniqueClassification']='temp'
            df['RefinedOrientation']='temp'
            df['WindowClassifications']='temp'
            
            for row in df.index:
    
                haplotype = df.at[row,'Haplotype']
    
                chrom_seq = ''.join(pysam.faidx(assemblyDirectory+sampleName+"_chrY.fa.gz", haplotype+":"+str(df.at[row,'StartCoordinate'])+"-"+str(df.at[row,'EndCoordinate'])).split()[1:])
                if 'N' in chrom_seq:
                    NgapList.append("NGAP_PRESENT")
                else:
                    NgapList.append("NO_NGAP_PRESENT")
                acceptableColorRegions = [x for x in ast.literal_eval(str(df.at[row,'UniqueKmerProfileHitsLength'])).keys()]
                wins, new_start, new_end, un_start, un_end = classify_sequence_with_kmers_fast(chrom_seq, Colorkmers, k, acceptableColorRegions, int(df.at[row,'StartCoordinate']), int(df.at[row,'EndCoordinate']))
    
    
                colorList = []
                uniqueColorList=[]
                orientations=[]
                for window in wins:
                    for color in window[2].split("_"):
                        colorList.append(color)
                
                    if len(window[2].split("_"))==1 and len(window[2])>1:
                        uniqueColorList.append(window[2])
                
                    orientations.append(window[3])
    
                colorList2 = [c for c in colorList if c != '']
                orientations2 = [c for c in orientations if c != '']
    
                if len(uniqueColorList)<1:
                    uniqueColorList.append(max(collections.Counter(colorList2), key=collections.Counter(colorList2).get))
                else:
                    pass
    
        
                df.at[row,'RefinedStart'] = new_start
                df.at[row,'RefinedEnd'] = new_end
                df.at[row,'RefinedClassification'] = max(collections.Counter(colorList2), key=collections.Counter(colorList2).get)
                df.at[row,'RefinedUniqueClassification'] = max(collections.Counter(uniqueColorList), key=collections.Counter(uniqueColorList).get)
                df.at[row,'RefinedOrientation'] = max(collections.Counter(orientations2), key=collections.Counter(orientations2).get)
                df.at[row,'WindowClassifications'] = wins

            df['GAP_TEST']=NgapList
            #df.to_csv("/LeeLab/HPRC/chromosomeY/Data/ColorBlockDataframes_refined/"+str(filename)+".refined.csv")   
              
        else:
            continue

100%|███████████████████████████████████████████████████████████████████████| 145/145 [19:17<00:00,  7.98s/it]


In [28]:
NgapList

['NO_NGAP_PRESENT']